In [10]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path(
    r"E:\BW\ML projects\loan_default_prediction\data\raw\loan_default_dataset.csv"
)

df = pd.read_csv(DATA_PATH)

# Clean column names
df.columns = df.columns.astype(str).str.strip()

# Remove the blank column only if you confirmed it is an unwanted index
df = df.loc[:, df.columns != ""]

# Separate features and target
X = df.drop(columns=["DEFAULT_STATUS", "CUSTOMER_ID"])
y = df["DEFAULT_STATUS"]

print("Dataset:", df.shape)
print("Features:", X.shape)
print("Target:", y.shape)

Dataset: (10, 14)
Features: (10, 12)
Target: (10,)


In [11]:
print("Target distribution:")
print(y.value_counts(dropna=False))

print("\nNumerical columns:")
numerical_cols = X.select_dtypes(include=["number"]).columns.tolist()
print(numerical_cols)

print("\nCategorical columns:")
categorical_cols = X.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()
print(categorical_cols)

Target distribution:
DEFAULT_STATUS
0    6
1    4
Name: count, dtype: int64

Numerical columns:
['AGE', 'INCOME', 'EMPLOYMENT_YEARS', 'LOAN_AMOUNT', 'LOAN_TERM', 'INTEREST_RATE', 'CREDIT_SCORE', 'PREVIOUS_DEFAULTS', 'NUMBER_OF_LOANS']

Categorical columns:
['GENDER', 'CITY', 'LOAN_TYPE']


In [13]:
from sklearn.model_selection import train_test_split

print("Examples per class:")
print(y.value_counts())

if y.nunique() < 2:
    raise ValueError("DEFAULT_STATUS must contain at least two classes.")

if y.value_counts().min() < 2:
    raise ValueError(
        "A target class has fewer than 2 examples. "
        "Add more data before making a stratified split."
    )

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining features:", X_train.shape)
print("Testing features:", X_test.shape)
print("\nTraining target distribution:")
print(y_train.value_counts())

Examples per class:
DEFAULT_STATUS
0    6
1    4
Name: count, dtype: int64

Training features: (8, 12)
Testing features: (2, 12)

Training target distribution:
DEFAULT_STATUS
0    5
1    3
Name: count, dtype: int64


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numerical_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [15]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (8, 20)
Processed testing shape: (2, 20)


In [16]:
print("Original features:", X.shape)
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nProcessed training data:", X_train_processed.shape)
print("Processed test data:", X_test_processed.shape)

print("\nMissing values in training target:", y_train.isna().sum())
print("Missing values in test target:", y_test.isna().sum())

print("\nPreprocessing pipeline:")
print(preprocessor)

Original features: (10, 12)
Training set: (8, 12)
Test set: (2, 12)

Processed training data: (8, 20)
Processed test data: (2, 20)

Missing values in training target: 0
Missing values in test target: 0

Preprocessing pipeline:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['AGE', 'INCOME', 'EMPLOYMENT_YEARS',
                                  'LOAN_AMOUNT', 'LOAN_TERM', 'INTEREST_RATE',
                                  'CREDIT_SCORE', 'PREVIOUS_DEFAULTS',
                                  'NUMBER_OF_LOANS']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                     

In [18]:
from pathlib import Path
import joblib

PROJECT_ROOT = Path.cwd().parent  # If the notebook runs from notebooks/
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(preprocessor, MODEL_DIR / "preprocessor.joblib")

print("Saved to:", MODEL_DIR / "preprocessor.joblib")

Saved to: e:\BW\ML projects\loan_default_prediction\models\preprocessor.joblib
